In [1]:
# drive allocation
import os
from google.colab import drive
from datasets import load_dataset, Dataset, load_from_disk
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
BASE_PATH = r"CSCI 5980 8980 Project"

ORIGIN_PATH = f"/content/drive/MyDrive/{BASE_PATH}/Notebooks/MICE_Output/"
OUTPUT_PATH = f"/content/drive/MyDrive/{BASE_PATH}/Notebooks/MICE_Output/eval-1k/"

layers_full_path = os.path.join(OUTPUT_PATH, "layers_full.csv")

summary_full_path = os.path.join(OUTPUT_PATH, "summary_full.csv")

In [3]:
summaryA = os.path.join(ORIGIN_PATH, "lite_results_summary_judged_trek.csv")
summaryB  = os.path.join(ORIGIN_PATH, "results_summary_judged_brandon.csv")

layersA = os.path.join(ORIGIN_PATH, "lite_results_layers_trek.csv")
layersB = os.path.join(ORIGIN_PATH, "results_layers_brandon.csv")

In [4]:
summary_dfA = pd.read_csv(summaryA)
summary_dfB = pd.read_csv(summaryB)

layers_dfA = pd.read_csv(layersA)
layers_dfB = pd.read_csv(layersB)

# combine dfA and dfB
summary_df = pd.concat([summary_dfA, summary_dfB])
layers_df = pd.concat([layers_dfA, layers_dfB])

summary_df.to_csv(summary_full_path, index=False)
layers_df.to_csv(layers_full_path, index=False)

In [5]:
# Pivot & Merge
layer_pivot = (
    layers_df.pivot_table(index="question_id", columns="layer", values="f1")
    .sort_index(axis=1)
    .reset_index()
)
layer_pivot.columns = ["question_id"] + [f"f1_layer_{c}" for c in layer_pivot.columns[1:]]

df = summary_df.merge(layer_pivot, on="question_id")

# Define Features and Stratification
layer_cols = [c for c in df.columns if c.startswith("f1_layer_")]
feature_cols = layer_cols + ["normalized_log_confidence"]

# Create a combined key for balanced 'type' AND 'judge_decision'
df['stratify_key'] = df['type'].astype(str) + "_" + df['judge_decision'].astype(str)

# Stratified Splitting
# Split 1: 80% Train+Val, 20% Test
df_tv, df_test = train_test_split(
    df, test_size=0.20, random_state=42, stratify=df['stratify_key']
)

# Split 2: Of that 80%, 25% for Val (results in 60% Train, 20% Val total)
df_train, df_val = train_test_split(
    df_tv, test_size=0.25, random_state=42, stratify=df_tv['stratify_key']
)

# Extract Arrays
def get_xyt(target_df):
    X = target_df[feature_cols].values.astype(np.float32)
    y = target_df["judge_decision"].values.astype(int)
    act_type = target_df["type"].values.astype(str)
    pred_type = target_df["predicted_type"].values.astype(str)
    return X, y, act_type, pred_type

X_train, y_train, type_train, ptype_train = get_xyt(df_train)
X_val, y_val, type_val, ptype_val = get_xyt(df_val)
X_test, y_test, type_test, ptype_test = get_xyt(df_test)

# Save Bundle with Joblib
data_bundle = {
    'train': (X_train, y_train, type_train, ptype_train),
    'val': (X_val, y_val, type_val, ptype_val),
    'test': (X_test, y_test, type_test, ptype_test),
    'feature_names': feature_cols,
    'metadata': {
        'train_ids': df_train['question_id'].values,
        'val_ids': df_val['question_id'].values,
        'test_ids': df_test['question_id'].values
    }
}

# preview X_test, y_train, z_test
print("X_test:", X_test[:5])
print("y_test:", y_test[:5])
print("type_test:", type_test[:5])
print("ptype_test:", ptype_test[:5])

bundle_path = os.path.join(OUTPUT_PATH, "data_bundle.joblib")
joblib.dump(data_bundle, bundle_path)

print(f"Data bundle saved successfully to: {bundle_path}")
print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

X_test: [[-0.31276906 -0.34874603 -0.36656728 -0.3887169  -0.35006192 -0.39508936
  -0.29004768 -0.25564334 -0.2876144  -0.31733438 -0.29830286 -0.37546626
  -0.29847836 -0.30709764 -0.4040911  -0.32134312 -0.3322961  -0.33720577
  -0.3208759  -0.23055401 -0.17400926 -0.02693143 -0.12333369 -0.00396424
   0.10632011  0.06067146 -0.00790447  0.11854129  0.1797379   0.4706413
   0.5735714  -0.16981964]
 [-0.6409685  -0.6244631  -0.6217413  -0.6218833  -0.5990439  -0.6079245
  -0.62808865 -0.60426575 -0.6196901  -0.5697827  -0.62504584 -0.62254333
  -0.594931   -0.63887    -0.64622813 -0.5967738  -0.58217585 -0.47833917
  -0.44525284 -0.38577753 -0.33470783 -0.23011327 -0.11762907  0.03037864
   0.13107571  0.20436777  0.21518512  0.31023124  0.44869334  0.48503512
   0.65026903 -0.16644365]
 [-0.5879229  -0.5830464  -0.55139357 -0.56285155 -0.54742223 -0.54954123
  -0.56196684 -0.5652283  -0.5566345  -0.5745306  -0.52737147 -0.563774
  -0.53997016 -0.5628074  -0.5212847  -0.55143344 -0.5